In [ ]:
# this worked in the end ***********************************
# bash zcat spark_hm3_joined.grm.gz | awk -F'\t' '$4 > 0.05' > bash_filtered_grm.txt
# 120gb memory requested for screen qrsh - didnt try less

In [8]:
install.packages('R.utils')

Installing package into ‘/u/home/a/afcarrol/R/x86_64-pc-linux-gnu-library/4.2.2’
(as ‘lib’ is unspecified)

also installing the dependencies ‘R.oo’, ‘R.methodsS3’




In [5]:
install.packages('magrittr')

Installing package into ‘/u/home/a/afcarrol/R/x86_64-pc-linux-gnu-library/4.2.2’
(as ‘lib’ is unspecified)



In [1]:
library(data.table)
library(utils)
library(magrittr)

In [ ]:
shm3 <- fread("spark_hm3_joined.grm.gz",nrows=10)

In [ ]:
shm3

In [8]:


# Define the file path of your gz file
file_path <- "spark_hm3_joined.grm.gz"

# Use pipe() to read the gz file and then read the top 10 rows with read.table()
data <- pipe(paste0("gzip -dc ", file_path, " | head -n 10 |")) %>% read.table(header = TRUE)

ERROR: Error in read.table(., header = TRUE): no lines available in input


In [2]:
# create a connection to the gzipped file
con <- gzfile("spark_hm3_joined.grm.gz")

# read the first 10 rows using read.table()
data <- read.table(con, header = TRUE, nrows = 10)

# close the connection
#close(con)

# inspect the data
head(data)


,X1,X1.1,X6.365160e.05,X9.818537e.01
,<int>,<int>,<dbl>,<dbl>
1,2,1,636516,-0.01437491
2,2,2,636516,1.01248300
3,3,1,636516,-0.01641038
4,3,2,636516,-0.00598105
5,3,3,636516,1.00355100
6,4,1,636516,-0.02381898


In [20]:
data[1,4]

[1] -0.01437491

In [ ]:
con <- gzfile("spark_hm3_joined.grm.gz")

num_lines <- sum(count.fields(con, sep = '\t') > 0)

In [19]:
row_counts <- count.fields(con)

# get the total number of rows
num_rows <- length(row_counts)

# close the connection
#close(con)

# print the number of rows
cat("The file has", num_rows, "rows.\n")

ERROR: Error in count.fields(con): invalid connection


In [15]:
num_rows

ERROR: Error in eval(expr, envir, enclos): object 'num_rows' not found


In [13]:
data

X1,X1.1,X6.365160e.05,X9.818537e.01
<int>,<int>,<dbl>,<dbl>
2,1,636516,-0.014374910
2,2,636516,1.012483000
3,1,636516,-0.016410380
3,2,636516,-0.005981050
3,3,636516,1.003551000
4,1,636516,-0.023818980
4,2,636516,0.005360249
4,3,636516,0.505529500
4,4,636516,1.002113000


In [22]:
library(data.table)

filename <- 'spark_hm3_joined.grm.gz'

chunksize <- 10 * 1024 * 1024  # 10 MB

dt <- data.table()

con <- gzfile(filename, 'rb')
while(!isIncomplete(con)) {
  lines <- readLines(con, n = chunksize)
  lines <- grep("^[^#]", lines, value = TRUE)  # exclude comment lines
  if(length(lines) > 0) {
    lines <- strsplit(lines, '\t')
    lines <- lapply(lines, function(x) as.numeric(x[4]) > 0.05)
    dt <- rbindlist(list(dt, lines))
  }
}

output_filename <- 'filtered_data.csv'
fwrite(dt, file = output_filename, sep = ',', quote = FALSE, row.names = FALSE)



In [ ]:
con

In [ ]:
num_lines <- sum(count.fields(con, sep = '\t') > 0)